# Fine-tune a bias generator for XHAIL (Colab)
Runtime → Change runtime type → **T4 GPU**. Reads the repo straight from your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/Research Engineering/refresh-xhail/xhail"
!pip install -q unsloth
!pip install -q -e .

In [ ]:
%cd experiments/bias_llm
# data/ is already generated; to regenerate: !python generate_tasks.py --n 2000 --seed 0
!wc -l data/*.jsonl

## Train (≈20–30 min on a T4)

In [ ]:
!python train_unsloth.py --model unsloth/Qwen2.5-1.5B-Instruct --epochs 2 --out checkpoints/qwen1.5b

## Predict: fine-tuned vs. few-shot base model

In [ ]:
!python predict.py --model checkpoints/qwen1.5b --tasks data/test.jsonl --out preds/ft_test.jsonl
!python predict.py --model checkpoints/qwen1.5b --tasks benchmarks --out preds/ft_bench.jsonl
!python predict.py --model unsloth/Qwen2.5-1.5B-Instruct --fewshot 3 --tasks data/test.jsonl --out preds/base_test.jsonl
!python predict.py --model unsloth/Qwen2.5-1.5B-Instruct --fewshot 3 --tasks benchmarks --out preds/base_bench.jsonl

## Score with XHAIL

In [ ]:
for run, tasks in [('ft_test','data/test.jsonl'), ('ft_bench','benchmarks'), ('base_test','data/test.jsonl'), ('base_bench','benchmarks')]:
    !python evaluate.py --tasks {tasks} --method file --pred preds/{run}.jsonl --name {run} | tail -7